In [1]:
import ee

# NOTA: "viirs-peru" es el ID del proyecto de Google Cloud vinculado a Earth Engine, el nombre fue ese pero es para todo EARTH ENGINE
# sirve como autenticación para CUALQUIER dataset de Earth Engine (WorldCover, SRTM,
# Sentinel-2, Open Buildings, etc.), no solo para VIIRS.

try:
    ee.Initialize(project="viirs-peru")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="viirs-peru")

print("Google Earth Engine listo")

Google Earth Engine listo


In [3]:
from pathlib import Path
import geopandas as gpd

def find_project_root(marker="requirements.txt"):
    path = Path.cwd()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"No se encontró '{marker}' subiendo desde {path}")

PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / "data"

# Cargar la geobase ya limpia (generada en 00a_geobase)
geobase = gpd.read_file(DATA / "clean" / "staging" / "geobase_distrital.gpkg")
print("Distritos en geobase:", len(geobase))

Distritos en geobase: 1889


### SRTM — Elevación del terreno (`USGS/SRTMGL1_003`)

**Qué es**
El SRTM (Shuttle Radar Topography Mission) es un modelo de elevación digital (DEM) global, generado por NASA/USGS a partir de una misión de radar en el año 2000. Cubre casi todo el planeta a una resolución espacial de ~30 metros por píxel. No es una serie temporal: es una sola "foto" de la topografía, porque el terreno no cambia año a año (a diferencia del gasto o la anemia, que sí varían).

**Qué se puede encontrar ahí**
- **Elevación (msnm)** por píxel — es la única variable nativa del dataset.
- A partir de la elevación se puede derivar **pendiente (slope)** con `ee.Terrain.slope()`, que mide qué tan inclinado es el terreno en cada punto (0° = plano, valores altos = muy empinado).

**Por qué es relevante para anemia**
No es un proxy indirecto — es directamente relevante por una razón clínica concreta: el protocolo del MINSA (siguiendo la guía de la OMS) **ajusta el punto de corte de hemoglobina para diagnosticar anemia según la altitud**, porque la hemoglobina sube naturalmente en zonas de mayor altura como mecanismo de adaptación a la menor presión de oxígeno. Sin controlar por altitud, el modelo podría confundir un efecto fisiológico normal con una diferencia real de prevalencia entre distritos.

Además, la pendiente del terreno es un buen proxy de accesibilidad: distritos con pendientes altas suelen tener viviendas más dispersas, caminos más difíciles y menor cobertura de seguimiento domiciliario para los programas de suplementación de hierro.

**Qué conviene extraer**
Por cada distrito (usando el polígono de `limite_distrital` con `reduceRegions`), dos estadísticos:

| Variable | Cómo se calcula | Para qué sirve |
|---|---|---|
| `elevacion_media` | Promedio de elevación (msnm) dentro del polígono distrital | Ajustar/controlar el punto de corte de hemoglobina por altitud |
| `pendiente_media` | Promedio de pendiente (°) dentro del polígono, derivado con `ee.Terrain.slope()` | Proxy de accesibilidad y dispersión del territorio |

**¿Por qué no hace falta más que esto?**

El SRTM no tiene versión por año — proviene de una única misión de radar realizada en el año 2000, y esa es la única medición que existe. A diferencia de Sentinel-2 (que sí tiene imágenes nuevas cada pocos días) o VIIRS (que sí tiene composites mensuales), no hay un "SRTM 2021" ni un "SRTM actual" que se pueda pedir en Earth Engine: el dataset es fijo, y así seguirá siendo salvo que la NASA lance una nueva misión de topografía.

Esto convierte a `elevacion_media` y `pendiente_media` en lo que en un panel de datos se llama una **variable time-invariant** (invariante en el tiempo): una característica del lugar, no del año. En la práctica, esto significa que en el join final estas dos columnas se unen solo por `ubigeo`, sin incluir `Año` en el merge — y su valor se repite igual en las cinco filas (2021 a 2025) de un mismo distrito, porque el terreno de un distrito no cambia de un año a otro. No es una redundancia ni un error: es exactamente cómo ya se comportan otras variables estáticas del proyecto, como `idh_2019` o `pct_pobreza_total` en la tabla `ubigeo`, que tampoco tienen una versión por año y también se repetirán igual en el join.

Por eso no hace falta pedir la elevación por año ni preocuparse por qué año usar: no existe esa variación en la fuente, así que un solo valor por distrito alcanza para las dos funciones que cumple esta variable (ajuste clínico del diagnóstico y proxy de accesibilidad). Si más adelante se necesita capturar heterogeneidad interna del distrito (ej. un distrito con zonas muy planas y muy empinadas a la vez), se puede agregar la desviación estándar de elevación como variable adicional — pero seguiría siendo una sola medición fija, no una serie por año.

### Cargar límite distrital como FeatureCollection de Earth Engine

Hasta ahora `limite_distrital` es un GeoDataFrame local (para hacer joins con pandas/geopandas).
Para calcular estadísticos zonales (`reduceRegions`) Earth Engine necesita esos mismos polígonos
como `ee.FeatureCollection`. Se sube una sola vez y se reutiliza para SRTM, y luego para las
demás fuentes de GEE (JRC Water, WorldCover, Sentinel-2).

### SRTM — elevación y pendiente media por distrito (todos los distritos del Perú)

**Qué se hace en esta etapa**

Se calcula, para cada uno de los 1889 distritos del Perú, un único valor resumen de elevación y
un único valor resumen de pendiente del terreno. El resultado es una tabla con una fila por
distrito — no una imagen, no un mapa: una tabla lista para unirse (por `ubigeo`) al resto de
variables del proyecto.

**De dónde sale el dato**

La fuente es el SRTM (`USGS/SRTMGL1_003`), un modelo de elevación digital de NASA/USGS a 30
metros de resolución. Es una sola medición fija del año 2000 — no existe una versión "por año",
así que este valor se calcula una sola vez y se repite igual en las cinco filas (2021–2025) de
cada distrito en el panel final, porque el terreno de un lugar no cambia de un año a otro (ver
nota sobre variables *time-invariant*).

**Qué son las "capas" en este contexto**

Cada fuente de datos de Earth Engine (elevación, pendiente, cobertura de suelo, agua, etc.) es
una *capa* distinta: una imagen independiente, del tamaño de todo el planeta, donde cada píxel
tiene un valor propio de esa variable. Elevación y pendiente son dos capas separadas — la
pendiente no viene incluida en el SRTM original, se calcula matemáticamente a partir de la
elevación con `ee.Terrain.slope()`, y después ambas capas se apilan en una sola imagen de dos
bandas para poder procesarlas juntas en un solo paso.

**Cómo se pasa de "millones de píxeles" a "un solo número por distrito": el reducer**

El SRTM tiene un píxel cada 30×30 metros. Un distrito de tamaño mediano puede contener decenas
o cientos de miles de esos píxeles, cada uno con su propio valor de elevación y de pendiente
(es justo la variación de colores que se ve al visualizar el mapa: verde donde el valor es bajo,
rojo donde es alto).

Para convertir todos esos píxeles en un solo número representativo del distrito, se usa un
**reducer** — la operación de Earth Engine que resume muchos valores en uno solo. El reducer
usado acá es `ee.Reducer.mean()`, que simplemente **promedia** todos los valores de los píxeles
que caen dentro del polígono de cada distrito. Es el mismo cálculo que haría `columna.mean()`
en pandas, con la diferencia de que aquí "la columna" son todos los píxeles contenidos dentro
de la forma geográfica del distrito, no filas de una tabla.

El promedio es la elección correcta para estas dos variables específicas: para elevación, porque
el ajuste clínico del punto de corte de hemoglobina por altitud (protocolo MINSA/OMS) se define
en función de la altitud típica de la población, no del punto más alto ni más bajo del distrito;
y para pendiente, porque sirve como una medida general de qué tan accidentado es el distrito en
conjunto, útil como proxy de accesibilidad.

**Un límite conocido del promedio**

En distritos con mucha variación interna — por ejemplo, uno con un valle plano y cerros
empinados a la vez — el promedio puede terminar representando un punto intermedio que no
describe bien a ninguna de las dos zonas reales del distrito. Esto se documenta como limitación
conocida y queda como posible extensión (agregar la desviación estándar de pendiente como
variable adicional, que mediría qué tan mixto es el terreno), sin ser indispensable para el
núcleo mínimo demostrable del proyecto.

**Resultado esperado**

Una tabla (`srtm_distrital`) de 1889 filas × 3 columnas: `ubigeo`, `elevacion_media` y
`pendiente_media`, lista para unirse al resto del maestro distrital solo por `ubigeo` (sin año).

In [4]:
import json
import pandas as pd

# ─────────────────────────────────────────────────────────────
# SRTM — elevación y pendiente media por distrito (por lotes)
# ─────────────────────────────────────────────────────────────

# Imagen de elevación + pendiente derivada, apiladas en una sola imagen
srtm_elevation = ee.Image("USGS/SRTMGL1_003").select("elevation")
srtm_slope = ee.Terrain.slope(srtm_elevation).rename("slope")
srtm_img = srtm_elevation.rename("elevation").addBands(srtm_slope)

# Base de distritos: solo ubigeo + geometry, geometría simplificada para no
# exceder el límite de payload de Earth Engine
distritos_base = (
    geobase[["UBIGEO", "geometry"]]          # <- antes decía limite_distrital
    .rename(columns={"UBIGEO": "ubigeo"})
    .copy()
)
distritos_base["geometry"] = distritos_base.geometry.simplify(
    tolerance=0.005, preserve_topology=True
)
distritos_base = distritos_base[
    distritos_base.geometry.notna() & ~distritos_base.geometry.is_empty
].copy()

print("Distritos a procesar:", len(distritos_base))

# Solo promedio — max/min no aportan al objetivo del proyecto (ver nota abajo)
reducer_srtm = ee.Reducer.mean()

chunk_size = 40 # partimos en 40 para q no exceda el límite de payload de Earth Engine (aprox 1MB por request)
resultados = []

for i in range(0, len(distritos_base), chunk_size):
    chunk = distritos_base.iloc[i:i + chunk_size].copy()
    print(f"Procesando distritos {i + 1} a {min(i + chunk_size, len(distritos_base))}...")

    geojson_chunk = json.loads(chunk.to_json())
    distritos_ee_chunk = ee.FeatureCollection(geojson_chunk)

    stats_chunk = srtm_img.reduceRegions(
        collection=distritos_ee_chunk,
        reducer=reducer_srtm,
        scale=30,       # resolución nativa del SRTM
        tileScale=4,    # evita timeouts en distritos grandes
    )
    stats_chunk = stats_chunk.map(lambda f: f.setGeometry(None))

    features = stats_chunk.getInfo()["features"]
    df_chunk = pd.DataFrame([f["properties"] for f in features])
    resultados.append(df_chunk)

# Unir todos los lotes
srtm_distrital = pd.concat(resultados, ignore_index=True)
srtm_distrital["ubigeo"] = srtm_distrital["ubigeo"].astype(str).str.zfill(6)

# Renombrar a los nombres finales que se usarán en el maestro distrital
srtm_distrital = srtm_distrital.rename(columns={
    "elevation": "elevacion_media",
    "slope": "pendiente_media",
})[["ubigeo", "elevacion_media", "pendiente_media"]]

# Validación
print("\nSRTM distrital listo")
print("Distritos con SRTM:", len(srtm_distrital))
print("UBIGEO únicos:", srtm_distrital["ubigeo"].nunique())
print("Duplicados:", srtm_distrital["ubigeo"].duplicated().sum())

srtm_distrital.head()

Distritos a procesar: 1889
Procesando distritos 1 a 40...
Procesando distritos 41 a 80...
Procesando distritos 81 a 120...
Procesando distritos 121 a 160...
Procesando distritos 161 a 200...
Procesando distritos 201 a 240...
Procesando distritos 241 a 280...
Procesando distritos 281 a 320...
Procesando distritos 321 a 360...
Procesando distritos 361 a 400...
Procesando distritos 401 a 440...
Procesando distritos 441 a 480...
Procesando distritos 481 a 520...
Procesando distritos 521 a 560...
Procesando distritos 561 a 600...
Procesando distritos 601 a 640...
Procesando distritos 641 a 680...
Procesando distritos 681 a 720...
Procesando distritos 721 a 760...
Procesando distritos 761 a 800...
Procesando distritos 801 a 840...
Procesando distritos 841 a 880...
Procesando distritos 881 a 920...
Procesando distritos 921 a 960...
Procesando distritos 961 a 1000...
Procesando distritos 1001 a 1040...
Procesando distritos 1041 a 1080...
Procesando distritos 1081 a 1120...
Procesando distritos

,ubigeo,elevacion_media,pendiente_media
0,010101,2423.265986,21.252837
1,010102,2819.011116,19.819781
2,010103,2332.129471,30.776104
3,010104,2525.921415,20.679454
4,010105,2626.557058,22.795578


In [5]:
output_path = DATA / "clean" / "staging" / "srtm_distrital.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

srtm_distrital.to_csv(output_path, index=False)
print("Guardado en:", output_path)

Guardado en: c:\Users\JHOSSEP\Documents\REPO\causal-anemia-model\data\clean\staging\srtm_distrital.csv
